## 4.5 Lunar Lander

**(a)**

The code computes the advantage estimate as:

```python
advantages = returns - values
```

where `returns` are the Monte Carlo tail returns $G_t = \sum_{\tau=t}^{T} \gamma^{\tau-t} r_\tau$ computed from a single episode rollout via `compute_returns`, and `values` = $V_w^\pi(x_t)$ are the critic's learned estimates from the `value_head` of the shared network. This corresponds to the advantage estimator:

$$\delta^\pi = G_t - V_w^\pi(x_t)$$

On the bias-variance spectrum, this estimator is **low bias, high variance**. It is unbiased since $G_t$ is a Monte Carlo estimate of $Q^\pi(x_t, u_t)$, so the advantage estimate is an unbiased estimate of $A^\pi(x_t, u_t)$. However, it has high variance since it relies on the full trajectory return, which accumulates randomness across all timesteps. Subtracting the baseline $V_w^\pi(x_t)$ reduces variance without introducing bias, because the baseline depends only on the state and not on the action, so it vanishes in expectation when computing the policy gradient.

**(b)**

Returns can vary by orders of magnitude across training, as early episodes may yield very negative returns while later ones are large and positive. Without standardization, gradient magnitudes would be inconsistent across episodes, making learning unstable. The code standardizes returns using a running EMA of the mean and variance:

```python
standardized_returns = (returns - return_ema) / (np.sqrt(return_emv) + 1e-6)
```

This keeps the critic targets and actor gradient weights at a consistent scale throughout training, similar in spirit to batch normalization, and improves learning stability.

**(c)**

`jax.lax.stop_gradient` prevents gradients from flowing through the advantage estimate when computing the actor loss:

```python
actor_loss = jnp.sum(-action_log_probs * jax.lax.stop_gradient(advantages) * mask)
```

Without it, the gradient of `actor_loss` would flow through `advantages = returns - values`, causing the actor update to also push the critic's `values` toward `returns`, effectively mixing actor and critic gradients through the shared trunk. This is incorrect: the policy gradient theorem requires the advantage to be treated as a fixed scalar weight $\delta^\pi$ when differentiating with respect to $\theta$. The critic already learns separately through `critic_loss = sum(advantages^2)`, so `stop_gradient` ensures the two losses remain decoupled as intended.

**(d)**

An alternative approach would be model-based trajectory optimization, such as iLQR or SCP (as in Problem 1). If the lunar lander dynamics were known or could be identified from data, one could linearize the dynamics at each timestep and solve a finite-horizon optimal control problem to compute a control sequence. Compared to the model-free A2C approach, trajectory optimization is significantly more sample efficient and can provide stronger guarantees on the quality of the solution. However, it requires an accurate dynamics model, since lunar lander contact forces and thruster dynamics are difficult to model precisely and model mismatch would introduce suboptimality or instability. The model-free RL approach here makes no assumptions on the dynamics and is more general, at the cost of requiring thousands of environment interactions to converge.